# End to end 4 — an experiment lands

A response surface was fitted to an observational panel in which units were dosed more
when something unobserved was also lifting the outcome. The amplitude it learned is
inflated. Then a randomized experiment lands: one number, one standard error, one
estimand that is *not* the decision's estimand. This notebook walks the evidence through
`axiom.calibrate` both ways and keeps every assumption on the ledger.

| step | subpackage |
|---|---|
| a world with a known truth and a confounded panel | `sim` |
| the experiment's estimand and the decision's | `estimands` |
| the experiment as a `Measurement` | `calibrate.evidence` |
| the biased fit, and two calibrated fits (prior route, likelihood route) | `surface`, `calibrate.prior`, `calibrate.likelihood` |
| does each fit agree with the experiment? | `calibrate.check` |
| reading the experiment as the decision estimand: plan, corrections, ledger | `estimands.transfer_to`, `calibrate.transfer`, `calibrate.ledger` |
| saving the whole thing without pickle | `io` |

The scenario is the one `tests/recovery/test_calibration_recovers_truth.py` asserts on,
with a carryover kernel added so the transfer has something to correct.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from axiom.calibrate import (
    Agreement, CalibratedSpec, Correction, Ledger, Measurement, ResolvedTransfer, aggregation_level,
    agreement, carryover_window_factor, derive_prior, fit_calibrated, resolve,
)
from axiom.core import Intervention, Population, Posterior, TimeWindow
from axiom.data import Panel
from axiom.estimands import Estimand, EstimandResult, Level, Quantity, RealizedDraws, TransferPlan, realize
from axiom.io import Analysis, load_analysis, save_analysis
from axiom.sim import DosePlan, surface_world
from axiom.surface import FitResult, GeometricCarryover, HillKernel, fit

pd.set_option("display.width", 160)

## The world and what the analyst sees

One treatment `a` with a Hill response (`beta_a = 10`, `k_a = 50`, `s_a = 2`) and a
geometric carryover (`lam_a = 0.5`, four lags): the dose accumulates first and saturates
second. The panel the analyst sees is the honest panel plus `u`, an unobserved confounder
correlated with the dose — exactly the situation in which a surface fit learns the wrong
amplitude and nothing inside the panel can tell it so.

In [ ]:
TRUTH = {"beta_a": 10.0, "alpha": 5.0, "k_a": 50.0, "s_a": 2.0, "lam_a": 0.5}
HI, LO = 100.0, 0.0
MAX_LAG = 4

world = surface_world(
    n_units=4, n_periods=24, treatments=("a",),
    kernels=HillKernel(reference_dose=50.0, amplitude_scale=10.0),
    carryover={"a": GeometricCarryover(max_lag=MAX_LAG)},
    doses=DosePlan(scale=50.0, spread=0.8, zero_fraction=0.05),
    intercept="shared", truth=TRUTH, noise_sd=2.0, seed=0,
)
spec = world.spec

rng = np.random.default_rng(123)
frame = world.panel.frame
dose = frame["a"].to_numpy(dtype=np.float64)
u = (dose - dose.mean()) / dose.std() + 0.3 * rng.standard_normal(dose.size)
observed = Panel(frame.assign(y=frame["y"] + 1.0 * u), world.panel.roles)
print(observed)
print("structural parameters:", world.structural)
print("correlation(dose, u) =", round(float(np.corrcoef(dose, u)[0, 1]), 3))

## Two estimands

The experiment was a one-period randomized contrast: dose `a = 100` against `a = 0`,
read in the first period, per unit. The decision needs the *steady-state per-period*
lift of the same contrast, per cluster (the panel's four units form one cluster). They
differ on the `window` and `level` facets; `transfer_to` will say so below.

In [ ]:
def contrast(name: str, window: TimeWindow, level: Level) -> Estimand:
    return Estimand(
        name=name, quantity=Quantity(kind="contrast"), treatment=spec.treatment("a"),
        intervention=Intervention(doses={"a": HI}), reference=Intervention(doses={"a": LO}),
        outcome=spec.outcome, population=Population(name="panel_units"), window=window, level=level,
        dimension=spec.outcome_dimension,
    )


experiment_estimand = contrast(
    "first_period_lift", TimeWindow(start=0, stop=1, basis="cumulative"), Level(unit="individual")
)
decision_estimand = contrast(
    "steady_state_lift", TimeWindow(start=MAX_LAG, stop=world.n_periods, basis="per_period"), Level(unit="cluster")
)
for e in (experiment_estimand, decision_estimand):
    print(f"{e.name:20s} window={e.window.start}..{e.window.stop} ({e.window.basis}) level={e.level.unit}")

## The experiment as a `Measurement`

The truth of the experiment's estimand comes from the world's one `forward` — the same
function the likelihood, the DGP, and the design math all call. The measurement is that
truth plus noise at a 1 % standard error, with its method, size, and source recorded.

In [ ]:
diff = world.forward({"a": HI}) - world.forward({"a": LO})  # (unit, period)
truth_experiment = float(np.mean(diff[:, 0]))
truth_decision = float(np.sum(np.mean(diff[:, MAX_LAG:], axis=1)))  # cluster level sums over units
se = 0.01 * truth_experiment
measurement = Measurement(
    estimand=experiment_estimand,
    estimate=truth_experiment + se * float(np.random.default_rng(7).standard_normal()),
    se=se, method="randomized_contrast", n_units=40, n_periods=1, source="rct-2026Q1",
)
print(f"truth of the experiment's estimand: {truth_experiment:.4f}; it reads {measurement.estimate:.4f} ± {measurement.se:.4f}")
print(f"truth of the decision's estimand:   {truth_decision:.4f}  (per cluster of {world.n_units} units)")
print(measurement.interval, "| target estimand hash", measurement.target[:12])

## The biased fit

A Laplace fit of the world's own spec to the confounded panel. The amplitude lands more
than two posterior standard deviations above the truth, and `agreement` says the fit
disagrees with the experiment — which is the only reason the analyst knows something is
wrong.

In [ ]:
def moments(result: FitResult) -> dict[str, tuple[float, float]]:
    assert isinstance(result.posterior, Posterior) and result.converged
    return {n: (float(result.posterior.flat(n).mean()), float(result.posterior.flat(n).std(ddof=1)))
            for n in ("beta_a", "k_a", "s_a", "lam_a", "alpha")}


def show(label: str, result: FitResult) -> None:
    m = moments(result)
    print(f"{label:18s}" + "  ".join(f"{n} {mu:7.3f}±{sd:5.3f}" for n, (mu, sd) in m.items()))


biased = fit(spec, observed, backend="laplace", draws=1000, seed=1)
print(" " * 18 + "  ".join(f"{n} truth {TRUTH[n]:<9}" for n in ("beta_a", "k_a", "s_a", "lam_a", "alpha")))
show("biased", biased)
b_mean, b_sd = moments(biased)["beta_a"]
print(f"amplitude sits {(b_mean - TRUTH['beta_a']) / b_sd:.1f} posterior sd above the truth")
check_biased = agreement(biased, measurement, seed=0)
assert isinstance(check_biased, Agreement)
print(f"agreement with the experiment: z = {check_biased.z:+.2f} -> {check_biased.verdict}")

## Route 1 — the prior route

`derive_prior` turns the measurement into a prior on the amplitude. It needs the *design
factor* — how many outcome units the measured contrast is per unit of amplitude — which it
estimates from paired draws of `beta_a` and the realized contrast under the biased fit.
It changes only the amplitude's prior; every other prior and the mean tree are untouched,
and the ledger lines it writes say what was assumed.

In [ ]:
realized = realize(experiment_estimand, biased, assume_identified=True, keep_draws=True)
assert isinstance(realized, RealizedDraws)
beta_draws = biased.posterior.flat("beta_a")
contribution_draws = np.asarray(realized.draws, dtype=np.float64).reshape(-1)
calibrated = derive_prior([measurement], spec, "a", beta_draws=beta_draws, contribution_draws=contribution_draws)
assert isinstance(calibrated, CalibratedSpec)
print(f"design factor {calibrated.design_factor:.4f} | implied amplitude {calibrated.amplitude_mean:.3f} ± {calibrated.amplitude_sd:.3f}")
print("new prior on", calibrated.parameter, "->", calibrated.prior)
for line in calibrated.ledger_lines:
    print(f"  ledger: {line.kind:<22} [{line.assumption.name}]")

prior_route = fit(calibrated.spec, observed, backend="laplace", draws=1000, seed=1)
show("prior route", prior_route)

## Route 2 — the likelihood route

`fit_calibrated` keeps the original priors and adds a soft constraint: the realized
contrast of the experiment's estimand, evaluated through `forward`, must sit within
`se` of the measurement. Because that contrast depends on `k_a`, `s_a`, and `lam_a` as
well as on the amplitude, the constraint pulls the curve shape too.

In [ ]:
likelihood_route = fit_calibrated(spec, observed, [measurement], backend="laplace", draws=1000, seed=1)
assert isinstance(likelihood_route, FitResult)
show("likelihood route", likelihood_route)
print("route:", likelihood_route.provenance["route"], "| constraint:", likelihood_route.provenance["constraints"][0]["name"])

## Does each fit agree with the experiment?

`agreement` realizes the experiment's estimand from each fit and compares it with the
measurement. The likelihood route agrees by construction of its constraint. The prior
route moves toward the measurement but may not reach `agrees`: its design factor was
computed under the biased curve shape, so the prior it derived is only part of the way
— the recovery test's docstring works through why.

In [ ]:
rows = []
for label, result in (("biased", biased), ("prior route", prior_route), ("likelihood route", likelihood_route)):
    ag = agreement(result, measurement, seed=0)
    assert isinstance(ag, Agreement)
    rows.append({"fit": label, "realized": ag.posterior_mean, "sd": ag.posterior_sd, "z": ag.z, "verdict": ag.verdict,
                 "beta_a": moments(result)["beta_a"][0]})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("measurement:", round(measurement.estimate, 3), "| truth:", round(truth_experiment, 3), "| beta_a truth:", TRUTH["beta_a"])

## Reading the experiment as the decision estimand

The calibrated fit can realize the decision estimand directly — that is what the one
`forward` buys. But a decision memo also wants the experiment's *own* number read on the
decision's terms, with every step licensed. `transfer_to` names the facets that differ;
each gets a `Correction` with its assumption; `resolve` combines them and writes the
ledger.

- **window**: `carryover_window_factor` scales a first-period read to the steady-state
  total using the carryover weights at the calibrated `lam_a`.
- **level**: `aggregation_level` reads an individual effect at cluster level (point
  factor 1 under additivity; the standard error is rescaled by the design effect).

Both lines are `unverified` — they are assumptions, and the world lets us check one of
them.

In [ ]:
plan: TransferPlan = experiment_estimand.transfer_to(decision_estimand)
print("plan:", plan.status, "| differing facets:", plan.differing)

lam_hat = float(likelihood_route.posterior.flat("lam_a").mean())
window = carryover_window_factor(GeometricCarryover(max_lag=MAX_LAG), {"lam_a": lam_hat}, 1, treatment="a")
level = aggregation_level(experiment_estimand.level, decision_estimand.level, cluster_size=world.n_units, icc=0.05)
assert isinstance(window, Correction) and isinstance(level, Correction)
print(f"window: share of carryover mass in period 0 = {window.counterfactual:.4f} -> factor {window.value:.4f} (at lam_a = {lam_hat:.3f})")
print(f"level:  point factor {level.detail['point_factor']}, SE factor {level.value:.4f}")

resolved: ResolvedTransfer = resolve(plan, corrections=[window, level])
per_member, per_member_se = resolved.apply(measurement.estimate, measurement.se)
print(f"\nresolved: {resolved.status} | licensed: {resolved.licensed} | factor {resolved.factor:.4f} se_scale {resolved.se_scale:.4f}")
print(f"experiment read as the decision estimand, per cluster member: {per_member:.3f} ± {per_member_se:.3f}")

### Checking the window assumption against the world

The window line assumes the share of the *effect* inside the window equals the share
of the *carryover weights* inside it. That is exact when the response is linear in the
accumulated dose. Here the dose accumulates *before* the Hill saturation, so the
first-period effect is `hill(w_0 · 100)`, not `w_0 · hill(100)`, and the linear factor
overshoots. The ledger calls the line `unverified` for exactly this reason. The
calibrated model's own reading of the decision estimand (through `forward`, no linear
assumption) is still off — its curve shape has not fully recovered from the confounding
— but it says so with a wide interval, whereas the transferred number carries a 0.5 %
standard error that hides the assumption entirely. That is what the ledger line is for.

In [ ]:
truth_ratio = truth_decision / world.n_units / truth_experiment
print(f"true steady-state / first-period ratio: {truth_ratio:.4f}   linear carryover factor: {window.value:.4f}")
decision_read = realize(decision_estimand, likelihood_route, assume_identified=True, seed=0)
assert isinstance(decision_read, EstimandResult)
print(f"decision estimand, truth per cluster: {truth_decision:.3f}")
print(f"  calibrated model through forward:   {decision_read.summary.mean:.3f} {decision_read.summary.interval}")
print(f"  experiment × transfer (per cluster): {per_member * world.n_units:.3f} ± {per_member_se * world.n_units:.3f}  <- overstated by the linear window assumption")

## The ledger

`resolve` returns a `Ledger`: one line per differing facet, each with its assumption,
its state, the counterfactual value (what you would have used without the correction)
and the value used. `Ledger.from_plan` is the baseline the plan alone implies — every
differing facet explicitly *uncorrected* — and `check_complete` says whether a ledger
covers the plan.

In [ ]:
ledger: Ledger = resolved.ledger
print(ledger.to_frame()[["facet", "assumption", "state", "counterfactual", "value", "correction"]].to_string())
print()
print(ledger.summary())
print()
baseline = Ledger.from_plan(plan)
print("plan-only baseline covers:", baseline.facets_covered(), "| complete:", baseline.check_complete(plan).status)
print("resolved ledger covers:   ", ledger.facets_covered(), "| complete:", ledger.check_complete(plan).status)

## Save it, load it, same numbers

Everything above is a `Spec`, a `Panel`, or a `Posterior`, so the whole analysis goes
into an `analysis.axiom` directory: JSON envelopes, a CSV, an `npz`. No pickle. After
reload every content hash matches and the numbers are bit-identical.

In [ ]:
analysis = (
    Analysis(specs={
        "surface": spec, "surface:calibrated": calibrated.spec, "calibration": calibrated,
        "estimand:experiment": experiment_estimand, "estimand:decision": decision_estimand,
        "result:decision": decision_read, "transfer": resolved, "ledger": ledger,
    })
    .with_panel(observed)
    .with_posterior(likelihood_route.posterior)
    .with_evidence(measurement)
    .with_ledger_line(*calibrated.ledger_lines, *ledger.lines)
)
root = Path(tempfile.mkdtemp()) / "experiment-lands.axiom"
saved = save_analysis(analysis, root, seed=1)
loaded = load_analysis(root)
print("equal after round-trip:", loaded == saved)
print("hashes identical:", loaded.hashes() == saved.hashes(), "|", len(loaded.hashes()), "hashed parts")
print("posterior beta_a mean identical:", float(loaded.posterior.flat("beta_a").mean()) == float(likelihood_route.posterior.flat("beta_a").mean()))
print("decision estimate identical:", loaded.spec("result:decision").summary.mean == decision_read.summary.mean)
print("no pickle on disk:", not any(p.suffix in (".pkl", ".pickle") for p in root.rglob("*")))
print(sorted(p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()))